In [1]:
import tsl
import torch
import numpy as np
import pandas as pd
from tsl.datasets import MetrLA, AirQuality
from einops import rearrange
from torch_geometric.utils.undirected import is_undirected
from tsl.engines import Imputer, Predictor
from torch_geometric.utils.loop import remove_self_loops
from torch_geometric.utils.isolated import contains_isolated_nodes
from tsl.data import SpatioTemporalDataset
from torch_geometric.utils import to_dense_adj, to_scipy_sparse_matrix
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from topomodelx.utils.sparse import from_sparse
from torch_sparse import SparseTensor
from torch_geometric.utils.sparse import to_edge_index
import toponetx as tnx
import networkx as nx
import torch
from torch_cluster import random_walk
import itertools
from utils.random_walk import uniform_random_walk, uniqueness
import torch.nn.functional as F
from tsl.nn.layers.recurrent.base import GraphGRUCellBase
from tsl.nn.blocks.encoders.recurrent.base import RNNBase
from tsl.nn.models import base_model
from tsl.nn import models
from tsl.metrics import numpy as numpy_metrics
from tsl.metrics import torch as torch_metrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tsl.data.preprocessing import StandardScaler, RobustScaler
from pytorch_lightning import Trainer
import math
import gc
import torch.nn as nn
from dataset_utils import SDWPE

import random
import torch
import numpy as np
import os

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.profilers import PyTorchProfiler,AdvancedProfiler
from pytorch_lightning.profilers import AdvancedProfiler

from torch.optim.lr_scheduler import MultiStepLR
from pytorch_lightning.loggers import TensorBoardLogger
from utils import MaskedRMSE



def seed_everything(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision('medium')  # 'medium' favors performance over precision

        # Enable TF32 format which is optimized for Tensor Cores on Ampere+ GPUs
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

seed_everything(44)

44

In [2]:
# dataset = MetrLA(root='./data/metrla')

# connectivity = dataset.get_connectivity(threshold=0.1,
#                                         include_self=False,
#                                         # normalize_axis=1,
#                                         force_symmetric=False,
#                                         layout="edge_index")

# covariates = {'u': dataset.datetime_encoded('day').values}

# torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
#                                       connectivity=connectivity,
#                                       mask=dataset.mask,
#                                       covariates=covariates,
#                                       horizon=12,
#                                       window=12,
#                                       stride=1)
# print(torch_dataset)

In [3]:
# dataset = AirQuality(root='./data/aq', impute_nans=True, small=False)

# splitting = {"val_len": 0.1,
#             "test_len": 0.2}


# connectivity_sparse= {"method": "distance",
#                     "threshold": 0.1,
#                     "include_self": False,
#                     "layout": "edge_index"}

# adj = dataset.get_connectivity(**connectivity_sparse)

# covariates = {'u': dataset.datetime_encoded('day').values}

# torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
#                                       connectivity=adj,
#                                       mask=dataset.mask,
#                                       covariates=covariates,
#                                       horizon=12,
#                                       window=12,
#                                       stride=1)

# torch_dataset

In [4]:
dataset = SDWPE()

splitting = {"val_len": 0.1,
            "test_len": 0.2}


connectivity_sparse= {"method": "distance",
                    "threshold": 0.1,
                    "include_self": False,
                    "layout": "edge_index"}

adj = dataset.get_connectivity(**connectivity_sparse)

covariates = {'u': dataset.datetime_encoded('day').values}

torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
                                      connectivity=adj,
                                      mask=dataset.mask,
                                      covariates=covariates,
                                      horizon=12,
                                      window=12,
                                      stride=1)

torch_dataset

SpatioTemporalDataset(n_samples=50093, n_nodes=134, n_channels=1)

In [5]:
# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=16,
    workers = 4
)

dm.setup()
print(dm)

{Train dataloader: size=36056}
{Validation dataloader: size=3995}
{Test dataloader: size=10018}
{Predict dataloader: None}


In [6]:
from tsl.nn.blocks.encoders import DCRNN, ConditionalBlock
from tsl.nn.blocks.encoders import DCRNN

# Inherit from your original DCRNNModel
class CustomDCRNNModel(models.DCRNNModel):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        # Replace only the dcrnn layer
        self.dcrnn = DCRNN(input_size=self.dcrnn.input_size,
                           hidden_size=self.dcrnn.hidden_size,
                           n_layers=len(self.dcrnn.cells),
                           k=self.dcrnn.k,
                           return_only_last_state=True,
                           root_weight=False,
                           add_backward=True)

In [7]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
        'mae': torch_metrics.MaskedMAE(),
        'rmse': MaskedRMSE(),
        # 'mae_step_2': torch_metrics.MaskedMAE(at=2),
        # 'mae_step_3': torch_metrics.MaskedMAE(at=5),
        # 'mae_step_4': torch_metrics.MaskedMAE(at=11),
        # 'mse_step_2': torch_metrics.MaskedMSE(at=2),
        # 'mse_step_3': torch_metrics.MaskedMSE(at=5),
        # 'mse_step_4': torch_metrics.MaskedMSE(at=11)
    }

model = CustomDCRNNModel(input_size=1,exog_size=2, hidden_size = 64, output_size=1,
                          horizon=12, ff_size = 128, dropout = 0.1,kernel_size=3,
                          cache_support=True, n_layers = 2)

def get_model_log_name(model, torch_dataset):
    class_name = model.__class__.__name__
    directed = str(not is_undirected(torch_dataset.edge_index))
    return f"{class_name}_directed_{directed}"
    

logger = TensorBoardLogger(
        save_dir=f"logs/{dataset.name}",
        name=get_model_log_name(model,torch_dataset)
)

In [8]:
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 1e-3,
                  'weight_decay':1e-4
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = False,
    # scheduler_class = MultiStepLR,
    # scheduler_kwargs = {'milestones':[40, 80, 120]}
)
# 'momentum':0.9,
#                  'nesterov':True

In [9]:
checkpoint_callback = ModelCheckpoint(
    dirpath=f'model_checkpoint/{dataset.name}/{model.__class__.__name__}',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
    verbose=True,
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=5,
        mode='min',
    min_delta = 0.001
    )

trainer = Trainer(
        max_epochs=200,
        limit_train_batches = 150,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        devices=[1],
        gradient_clip_val=5,
       callbacks=[early_stop_callback],
      # default_root_dir="logs",
        # profiler=profiler,
        # precision = '32',
        check_val_every_n_epoch = 3,
        logger=False

    
)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [10]:
trainer.fit(predictor, datamodule=dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | loss_fn       | MaskedMAE        | 0      | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
3 | test_metrics  | MetricCollection | 0      | train
4 | model         | CustomDCRNNModel | 313 K  | train
-----------------------------------------------------------
313 K     Trainable params
0         Non-trainable params
313 K     Total params
1.255     Total estimated model params size (MB)
49        Modules in train mode
0         Modules in eval mode


Training: |                                                                                           | 0/? [0…

Only args ['edge_index', 'x', 'u', 'edge_weight'] are forwarded to the model (CustomDCRNNModel).


Validation: |                                                                                         | 0/? [0…

Validation: |                                                                                         | 0/? [0…

...

Validation: |                                                                 | 0/? [00:00<?, ?it/s]

Validation: |                                                                 | 0/? [00:00<?, ?it/s]

Validation: |                                                                 | 0/? [00:00<?, ?it/s]

Validation: |                                                                 | 0/? [00:00<?, ?it/s]

Validation: |                                                                 | 0/? [00:00<?, ?it/s]

In [11]:
predictor.freeze()

trainer.test(ckpt_path="best", dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at /netfs/tsp/student/2022/zhu/ST_RUM/checkpoints/epoch=38-step=5850-v19.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /netfs/tsp/student/2022/zhu/ST_RUM/checkpoints/epoch=38-step=5850-v19.ckpt


Testing: |                                                                    | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │     91.7452621459961      │
│         test_mae          │     94.38578796386719     │
│         test_rmse         │     164.1033935546875     │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 94.38578796386719,
  'test_rmse': 164.1033935546875,
  'test_loss': 91.7452621459961}]